# Jaffle Shop: From Raw Data to Semantic Layer

This notebook demonstrates how to go from **raw synthetic data** to a fully auto-ingested **SLayer semantic layer** backed by DuckDB.

We'll:
1. Generate 3 years of realistic Jaffle Shop data using `jafgen`
2. Load it into DuckDB with proper schemas and foreign keys
3. Auto-ingest the database with SLayer, which discovers tables, relationships, and builds rollup joins
4. Explore the resulting models and run queries through the semantic layer

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys
import tempfile

import duckdb

# Add project root to path for local development
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))

from jaffle_shop_duckdb import SCHEMA_FILE, create_schema, generate_data, load_data

from slayer.core.models import DatasourceConfig
from slayer.core.query import Field, SlayerQuery
from slayer.engine.ingestion import ingest_datasource
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.storage.yaml_storage import YAMLStorage

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## Step 1: Generate Data & Load into DuckDB

[jafgen](https://github.com/dbt-labs/jaffle-shop-generator) simulates a chain of Jaffle shops over time, producing realistic data with seasonality, customer personas, and store openings. We generate 3 years of data and load it into DuckDB.

The schema has 7 tables with foreign key relationships:

```
customers  <--  orders  -->  stores
                  |
            order_items  -->  products  <--  supplies
                  
customers  <--  tweets
```

In [2]:
work_dir = tempfile.mkdtemp(prefix="jaffle_shop_")
db_path = os.path.join(work_dir, "jaffle_shop.duckdb")

# Generate CSV data (this takes ~45 seconds for 3 years)
data_dir = generate_data(work_dir, years=3)

# Create DuckDB database with schema and load data
conn = duckdb.connect(db_path)
create_schema(conn, SCHEMA_FILE)
load_data(conn, data_dir)

🥪 Pressing 1095 days of fresh jaffles... ━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:01:43% 0:00:010:00:05
🚚 Delivering jaffles... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:01 86% 0:00:01:--
  customers: 1926 rows
  stores: 6 rows
  products: 10 rows
  orders: 213424 rows
  order_items: 339824 rows
  supplies: 65 rows
  tweets: 108788 rows


Let's peek at what we loaded:

In [3]:
for table in ["customers", "stores", "products", "orders", "order_items", "supplies", "tweets"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:>15}: {count:>8,} rows")

print("\nSample orders:")
conn.execute("SELECT * FROM orders LIMIT 5").df()

      customers:    1,926 rows
         stores:        6 rows
       products:       10 rows
         orders:  213,424 rows
    order_items:  339,824 rows
       supplies:       65 rows
         tweets:  108,788 rows

Sample orders:


,id,customer_id,ordered_at,store_id,subtotal,tax_paid,order_total
0,a3f73624-2d61-4244-879f-0d97df133148,a16c4cfc-6507-456d-a60b-a3cda24589de,2018-09-01,8a623c96-5169-41b5-8b53-e5955de0bde3,95.0,5.70,100.70
1,a7fcb275-9929-4d93-b9d6-28a947212b1e,364d6fa8-7256-45cd-b6a0-1747f975ee8d,2018-09-02,8a623c96-5169-41b5-8b53-e5955de0bde3,9.0,0.54,9.54
2,b9ac9964-a9b3-457f-9e0b-0222706fe84c,2b766ebd-3416-40ac-94b9-6f04b8ff4d3d,2018-09-02,8a623c96-5169-41b5-8b53-e5955de0bde3,51.0,3.06,54.06
3,61971d28-1410-4e7f-8abb-634325bb1239,21a5d266-279a-40df-80a8-55c2f518c966,2018-09-02,8a623c96-5169-41b5-8b53-e5955de0bde3,14.0,0.84,14.84
4,b41e7827-5202-4ca8-8876-b7f50ee4a02f,92824e34-561e-47c2-84c9-a331e7f10b2d,2018-09-02,8a623c96-5169-41b5-8b53-e5955de0bde3,90.0,5.40,95.40


In [4]:
conn.close()

## Step 2: Auto-Ingest with SLayer

SLayer's `ingest_datasource()` introspects the database to discover:
- All tables and their column types
- Foreign key relationships between tables
- Transitive FK chains (e.g. `order_items` -> `orders` -> `customers`)

For each table it automatically generates:
- **Dimensions** from all columns (own + rolled-up from referenced tables)
- **Measures** like `count`, `sum`, `avg` for numeric columns
- **Rollup SQL** with LEFT JOINs for tables that reference others

In [5]:
# Configure the datasource and storage
storage = YAMLStorage(base_dir=os.path.join(work_dir, "slayer_models"))
ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=db_path)
storage.save_datasource(ds)

# Auto-ingest all tables
models = ingest_datasource(datasource=ds)

# Set default time dimensions for time-series queries
for model in models:
    if model.name == "orders":
        model.default_time_dimension = "ordered_at"
    elif model.name == "tweets":
        model.default_time_dimension = "tweeted_at"
    storage.save_model(model)

print(f"Discovered {len(models)} models:")
for m in sorted(models, key=lambda m: m.name):
    rollup = " (rollup)" if m.sql else ""
    print(f"  {m.name}: {len(m.dimensions)} dimensions, {len(m.measures)} measures{rollup}")

Discovered 7 models:
  customers: 2 dimensions, 1 measures
  order_items: 20 dimensions, 17 measures (rollup)
  orders: 13 dimensions, 11 measures (rollup)
  products: 5 dimensions, 3 measures
  stores: 4 dimensions, 3 measures
  supplies: 10 dimensions, 6 measures (rollup)
  tweets: 6 dimensions, 2 measures (rollup)


Let's inspect the **orders** model in detail. Notice how SLayer automatically rolled up dimensions from `customers` and `stores` via foreign keys:

In [6]:
orders_model = next(m for m in models if m.name == "orders")

print("Dimensions:")
for dim in orders_model.dimensions:
    pk = " [PK]" if dim.primary_key else ""
    print(f"  {dim.name} ({dim.type}){pk}")

print(f"\nMeasures:")
for meas in orders_model.measures:
    sql_info = f" sql={meas.sql}" if meas.sql else ""
    print(f"  {meas.name} ({meas.type}){sql_info}")

print(f"\nGenerated SQL:\n{orders_model.sql}")

Dimensions:
  id (string) [PK]
  customer_id (string)
  ordered_at (date)
  store_id (string)
  subtotal (number)
  tax_paid (number)
  order_total (number)
  customers__id (string)
  customers__name (string)
  stores__id (string)
  stores__name (string)
  stores__opened_at (date)
  stores__tax_rate (number)

Measures:
  count (count)
  subtotal_sum (sum) sql=subtotal
  subtotal_avg (avg) sql=subtotal
  tax_paid_sum (sum) sql=tax_paid
  tax_paid_avg (avg) sql=tax_paid
  order_total_sum (sum) sql=order_total
  order_total_avg (avg) sql=order_total
  stores__tax_rate_sum (sum) sql=stores__tax_rate
  stores__tax_rate_avg (avg) sql=stores__tax_rate
  customers__count (count_distinct) sql=customers__id
  stores__count (count_distinct) sql=stores__id

Generated SQL:
SELECT
    orders.id AS id,
    orders.customer_id AS customer_id,
    orders.ordered_at AS ordered_at,
    orders.store_id AS store_id,
    orders.subtotal AS subtotal,
    orders.tax_paid AS tax_paid,
    orders.order_total AS 

## Step 3: Query Through the Semantic Layer

Now we can query the data using SLayer's semantic API. Instead of writing SQL, we describe **what** we want: measures, dimensions, filters, and time granularity. SLayer generates and executes the SQL.

In [7]:
engine = SlayerQueryEngine(storage=storage)

### Revenue by store

Using the auto-rolled-up `stores__name` dimension from the orders model:

In [8]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    fields=[Field(formula="count"), Field(formula="order_total_sum")],
    dimensions=[{"name": "stores__name"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
))

for row in result.data:
    store = row["orders.stores__name"]
    count = row["orders.count"]
    revenue = row["orders.order_total_sum"]
    print(f"  {store}: {count:,} orders, ${revenue:,.2f} revenue")

  Brooklyn: 84,418 orders, $1,100,317.17 revenue
  Philadelphia: 62,491 orders, $841,244.77 revenue
  Chicago: 34,954 orders, $452,005.72 revenue
  San Francisco: 27,741 orders, $370,209.61 revenue
  New Orleans: 3,820 orders, $52,844.40 revenue


### Monthly order trends with cumulative sum

SLayer supports time dimensions with granularity, plus formula functions like `cumsum`:

In [9]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        Field(formula="count"),
        Field(formula="cumsum(count)", name="cumulative"),
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
))

print(f"{'Month':<12} {'Orders':>8} {'Cumulative':>12}")
print("-" * 34)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    count = row["orders.count"]
    cumulative = row["orders.cumulative"]
    print(f"{month:<12} {count:>8,} {cumulative:>12,}")

Month          Orders   Cumulative
----------------------------------
2018-09           426          426
2018-10           798        1,224
2018-11         1,033        2,257
2018-12         1,294        3,551
2019-01         1,490        5,041
2019-02         1,428        6,469
2019-03         2,497        8,966
2019-04         3,067       12,033
2019-05         3,449       15,482
2019-06         2,850       18,332
2019-07         1,554       19,886
2019-08         1,668       21,554
2019-09         2,803       24,357
2019-10         5,677       30,034
2019-11         5,625       35,659
2019-12         6,175       41,834
2020-01         6,350       48,184
2020-02         5,959       54,143
2020-03         6,365       60,508
2020-04         6,201       66,709
2020-05         7,766       74,475
2020-06         6,129       80,604
2020-07         3,036       83,640
2020-08         3,285       86,925
2020-09         5,945       92,870
2020-10        11,285      104,155
2020-11        11,61

### Top products by number of orders

This query traverses the `order_items` -> `products` rollup join automatically:

In [10]:
result = engine.execute(query=SlayerQuery(
    model="order_items",
    fields=[Field(formula="count"), Field(formula="quantity_sum")],
    dimensions=[{"name": "products__name"}, {"name": "products__type"}],
    order=[{"column": {"name": "quantity_sum"}, "direction": "desc"}],
))

print(f"{'Product':<35} {'Type':<10} {'Line Items':>12} {'Qty Sold':>10}")
print("-" * 70)
for row in result.data:
    name = row["order_items.products__name"]
    ptype = row["order_items.products__type"]
    count = row["order_items.count"]
    qty = row["order_items.quantity_sum"]
    print(f"{name:<35} {ptype:<10} {count:>12,} {qty:>10,}")

Product                             Type         Line Items   Qty Sold
----------------------------------------------------------------------
adele-ade                           BEVERAGE         47,930     49,864
chai and mighty                     BEVERAGE         47,867     49,782
vanilla ice                         BEVERAGE         47,579     49,502
tangaroo                            BEVERAGE         47,568     49,502
for richer or pourover              BEVERAGE         47,331     49,303
nutellaphone who dis?               JAFFLE           20,495     21,655
mel-bun                             JAFFLE           20,393     21,497
flame impala                        JAFFLE           20,366     21,481
the krautback                       JAFFLE           20,269     21,393
doctor stew                         JAFFLE           20,026     21,037


### Month-over-month revenue change

SLayer's `change()` function computes the difference from the previous period using a self-join (no edge NULLs, gap-safe):

In [11]:
result = engine.execute(query=SlayerQuery(
    model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        Field(formula="order_total_sum"),
        Field(formula="change(order_total_sum)", name="mom_change"),
        Field(formula="change_pct(order_total_sum)", name="mom_pct"),
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
    limit=12,
))

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12} {'MoM %':>8}")
print("-" * 48)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12} {pct_str:>8}")

Month             Revenue   MoM Change    MoM %
------------------------------------------------
2018-09      $   6,228.11            -        -
2018-10      $  11,123.72      $+4,896   +78.6%
2018-11      $  14,131.83      $+3,008   +27.0%
2018-12      $  18,276.25      $+4,144   +29.3%
2019-01      $  20,295.42      $+2,019   +11.0%
2019-02      $  19,248.15      $-1,047    -5.2%
2019-03      $  34,088.89     $+14,841   +77.1%
2019-04      $  40,009.01      $+5,920   +17.4%
2019-05      $  44,800.40      $+4,791   +12.0%
2019-06      $  38,975.71      $-5,825   -13.0%
2019-07      $  21,913.58     $-17,062   -43.8%
2019-08      $  23,821.32      $+1,908    +8.7%


## Summary

Starting from raw CSV data, we:
1. **Generated** 3 years of realistic e-commerce data with `jafgen`
2. **Loaded** it into DuckDB with proper schemas and foreign keys
3. **Auto-ingested** 7 tables into SLayer, which discovered all FK relationships and built rollup joins
4. **Queried** the semantic layer using high-level constructs (measures, dimensions, formulas) instead of raw SQL

SLayer handled the complexity of multi-table joins, time granularity, and analytical functions like cumulative sums and period-over-period comparisons -- all from a simple declarative query API.